In [67]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(497128, 31)


In [68]:
!pip install mlxtend

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [69]:
baskets = (
    df.groupby("orderNumber")
    ["skuNumber"]
    .apply(list)
    .tolist()
)

len(baskets)

177340

In [70]:
from mlxtend.preprocessing import (
    TransactionEncoder
)

te = TransactionEncoder()

basket_matrix = te.fit(
    baskets
).transform(
    baskets
)

basket_df = pd.DataFrame(
    basket_matrix,
    columns=te.columns_
)

basket_df.shape

(177340, 252)

In [71]:
from mlxtend.frequent_patterns import (
    fpgrowth
)

frequent_items = fpgrowth(
    basket_df,
    min_support=0.01,
    use_colnames=True
)

frequent_items.head()

,support,itemsets
0,0.013539,frozenset({SKUBV8})
1,0.458453,frozenset({SKU00060})
2,0.447034,frozenset({SKU00086})
3,0.132480,frozenset({SKU00084})
4,0.032401,frozenset({SKU500})


In [72]:
from mlxtend.frequent_patterns import (
    association_rules
)

rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.2
)

rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({SKU00086}),frozenset({SKU00060}),0.447034,0.458453,0.437239,0.978089,2.133458,1.0,0.232295,24.716319,0.960778,0.933778,0.959541,0.965909
1,frozenset({SKU00060}),frozenset({SKU00086}),0.458453,0.447034,0.437239,0.953728,2.133458,1.0,0.232295,11.950358,0.981036,0.933778,0.916320,0.965909
2,frozenset({SKU00084}),frozenset({SKU00060}),0.132480,0.458453,0.055216,0.416787,0.909117,1.0,-0.005520,0.928559,-0.103327,0.103069,-0.076938,0.268614
3,frozenset({SKU00084}),frozenset({SKU00086}),0.132480,0.447034,0.054489,0.411297,0.920057,1.0,-0.004734,0.939295,-0.091040,0.103783,-0.064629,0.266593
4,frozenset({SKUPM5}),frozenset({SKU00084}),0.139106,0.132480,0.040098,0.288257,2.175850,1.0,0.021669,1.218866,0.627730,0.173219,0.179565,0.295465


In [73]:
rules = (
    rules.sort_values(
        "lift",
        ascending=False
    )
)

rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].head(20)

,antecedents,consequents,support,confidence,lift
38,"frozenset({SKU00086, SKU00033})","frozenset({SKU00060, SKU00084})",0.010539,0.437193,7.917872
39,"frozenset({SKU00033, SKU00060})","frozenset({SKU00086, SKU00084})",0.010539,0.425450,7.808054
66,frozenset({SKU612}),frozenset({SKU613}),0.018620,0.415607,7.401466
65,frozenset({SKU613}),frozenset({SKU612}),0.018620,0.331593,7.401466
22,"frozenset({SKU00060, SKU00084})","frozenset({SKU00086, SKUPM5})",0.019804,0.358660,6.555843
19,"frozenset({SKU00086, SKUPM5})","frozenset({SKU00060, SKU00084})",0.019804,0.361987,6.555843
20,"frozenset({SKU00086, SKU00084})","frozenset({SKU00060, SKUPM5})",0.019804,0.363448,6.469976
21,"frozenset({SKU00060, SKUPM5})","frozenset({SKU00086, SKU00084})",0.019804,0.352540,6.469976
52,frozenset({SKU00303}),frozenset({SKU00010}),0.014204,0.214529,3.718196
53,frozenset({SKU00010}),frozenset({SKU00303}),0.014204,0.246188,3.718196


In [74]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName"
        ]
    ]
    .drop_duplicates()
)

sku_dict = dict(
    zip(
        sku_lookup["skuNumber"],
        sku_lookup["itemName"]
    )
)

In [75]:
rules["antecedent_names"] = (
    rules["antecedents"]
    .apply(
        lambda x:
        [
            sku_dict.get(i, i)
            for i in x
        ]
    )
)

rules["consequent_names"] = (
    rules["consequents"]
    .apply(
        lambda x:
        [
            sku_dict.get(i, i)
            for i in x
        ]
    )
)

In [76]:
rules[
    [
        "antecedent_names",
        "consequent_names",
        "confidence",
        "lift"
    ]
].head(20)

,antecedent_names,consequent_names,confidence,lift
38,"[Tulsi 00 (with Silver) 0.45g, Rajnigandha 100g ]","[Rajnigandha 4g, Tulsi 00 4.25g (6 Pcs)]",0.437193,7.917872
39,"[Rajnigandha 100g , Rajnigandha 4g]","[Tulsi 00 (with Silver) 0.45g, Tulsi 00 4.25g ...",0.425450,7.808054
66,[Center Fruit],[Center Fresh],0.415607,7.401466
65,[Center Fresh],[Center Fruit],0.331593,7.401466
22,"[Rajnigandha 4g, Tulsi 00 4.25g (6 Pcs)]","[Tulsi 00 (with Silver) 0.45g, Rajnigandha 17g...",0.358660,6.555843
19,"[Tulsi 00 (with Silver) 0.45g, Rajnigandha 17g...","[Rajnigandha 4g, Tulsi 00 4.25g (6 Pcs)]",0.361987,6.555843
20,"[Tulsi 00 (with Silver) 0.45g, Tulsi 00 4.25g ...","[Rajnigandha 4g, Rajnigandha 17g 2 Pcs]",0.363448,6.469976
21,"[Rajnigandha 4g, Rajnigandha 17g 2 Pcs]","[Tulsi 00 (with Silver) 0.45g, Tulsi 00 4.25g ...",0.352540,6.469976
52,[RG Pearl Elaichi 1g Hanger],[Pass Pass Meetha Magic 11.75g Hanger],0.214529,3.718196
53,[Pass Pass Meetha Magic 11.75g Hanger],[RG Pearl Elaichi 1g Hanger],0.246188,3.718196


In [77]:
print(rules.shape)

rules[
    [
        "antecedent_names",
        "consequent_names",
        "confidence",
        "lift"
    ]
].head(20)

(92, 16)


,antecedent_names,consequent_names,confidence,lift
38,"[Tulsi 00 (with Silver) 0.45g, Rajnigandha 100g ]","[Rajnigandha 4g, Tulsi 00 4.25g (6 Pcs)]",0.437193,7.917872
39,"[Rajnigandha 100g , Rajnigandha 4g]","[Tulsi 00 (with Silver) 0.45g, Tulsi 00 4.25g ...",0.425450,7.808054
66,[Center Fruit],[Center Fresh],0.415607,7.401466
65,[Center Fresh],[Center Fruit],0.331593,7.401466
22,"[Rajnigandha 4g, Tulsi 00 4.25g (6 Pcs)]","[Tulsi 00 (with Silver) 0.45g, Rajnigandha 17g...",0.358660,6.555843
19,"[Tulsi 00 (with Silver) 0.45g, Rajnigandha 17g...","[Rajnigandha 4g, Tulsi 00 4.25g (6 Pcs)]",0.361987,6.555843
20,"[Tulsi 00 (with Silver) 0.45g, Tulsi 00 4.25g ...","[Rajnigandha 4g, Rajnigandha 17g 2 Pcs]",0.363448,6.469976
21,"[Rajnigandha 4g, Rajnigandha 17g 2 Pcs]","[Tulsi 00 (with Silver) 0.45g, Tulsi 00 4.25g ...",0.352540,6.469976
52,[RG Pearl Elaichi 1g Hanger],[Pass Pass Meetha Magic 11.75g Hanger],0.214529,3.718196
53,[Pass Pass Meetha Magic 11.75g Hanger],[RG Pearl Elaichi 1g Hanger],0.246188,3.718196


In [78]:
rules[
    [
        "antecedent_names",
        "consequent_names",
        "confidence",
        "lift"
    ]
].head(20)

,antecedent_names,consequent_names,confidence,lift
38,"[Tulsi 00 (with Silver) 0.45g, Rajnigandha 100g ]","[Rajnigandha 4g, Tulsi 00 4.25g (6 Pcs)]",0.437193,7.917872
39,"[Rajnigandha 100g , Rajnigandha 4g]","[Tulsi 00 (with Silver) 0.45g, Tulsi 00 4.25g ...",0.425450,7.808054
66,[Center Fruit],[Center Fresh],0.415607,7.401466
65,[Center Fresh],[Center Fruit],0.331593,7.401466
22,"[Rajnigandha 4g, Tulsi 00 4.25g (6 Pcs)]","[Tulsi 00 (with Silver) 0.45g, Rajnigandha 17g...",0.358660,6.555843
19,"[Tulsi 00 (with Silver) 0.45g, Rajnigandha 17g...","[Rajnigandha 4g, Tulsi 00 4.25g (6 Pcs)]",0.361987,6.555843
20,"[Tulsi 00 (with Silver) 0.45g, Tulsi 00 4.25g ...","[Rajnigandha 4g, Rajnigandha 17g 2 Pcs]",0.363448,6.469976
21,"[Rajnigandha 4g, Rajnigandha 17g 2 Pcs]","[Tulsi 00 (with Silver) 0.45g, Tulsi 00 4.25g ...",0.352540,6.469976
52,[RG Pearl Elaichi 1g Hanger],[Pass Pass Meetha Magic 11.75g Hanger],0.214529,3.718196
53,[Pass Pass Meetha Magic 11.75g Hanger],[RG Pearl Elaichi 1g Hanger],0.246188,3.718196


In [79]:
rules_save = rules.copy()

for col in [
    "antecedents",
    "consequents",
    "antecedent_names",
    "consequent_names"
]:
    rules_save[col] = rules_save[col].astype(str)

rules_save.to_csv(
    "../data/processed/association_rules.csv",
    index=False
)

In [80]:
rules_save["antecedents"] = (
    rules_save["antecedents"]
    .astype(str)
)

rules_save["consequents"] = (
    rules_save["consequents"]
    .astype(str)
)

In [81]:
if "antecedent_names" in rules_save.columns:
    rules_save["antecedent_names"] = (
        rules_save["antecedent_names"]
        .astype(str)
    )

if "consequent_names" in rules_save.columns:
    rules_save["consequent_names"] = (
        rules_save["consequent_names"]
        .astype(str)
    )

In [82]:
rules_save = rules.copy()

In [83]:
rules_save["antecedents"] = (
    rules_save["antecedents"]
    .astype(str)
)

rules_save["consequents"] = (
    rules_save["consequents"]
    .astype(str)
)

rules_save["antecedent_names"] = (
    rules_save["antecedent_names"]
    .astype(str)
)

rules_save["consequent_names"] = (
    rules_save["consequent_names"]
    .astype(str)
)

In [84]:
rules_save.to_parquet(
    "../data/processed/association_rules.parquet",
    index=False
)

print("Saved successfully")

Saved successfully


In [85]:
print(rules.shape)

(92, 16)
